# Step 2: Warped Coordinate Prediction

Method Overview: 
0. Make sure to run assign_anchors.ipynb first to retreive the anchor image assignments first!
1. Estimates an anchor-to-test geometric transform and generates the coordinates

Be sure to specify if you would like to run evaluation
- Train
    - Find some transform between test and anchor image
    - Generate coordinates for each set of images
- Test
    - Apply the same transformation between test and DETERMINED anchor image from previous function
    - Generate coordinates for each set of images
    - EVALUATE: again the ground truth coordinates

Outputs created:
- correspondence/grouping.csv
- warped_coordinates/test_XX_warped_coordinates.csv (100 rows, x/y, 2 decimals)


In [1]:
from __future__ import annotations

import re
from pathlib import Path
from typing import Dict, List, Tuple

import cv2
import numpy as np
import pandas as pd

## Handling Image Files

In [ ]:
# Update these paths
anchors_dir = Path("/Volumes/LUCY DISK/mia/Project 1/example test/anchor_images")
tests_dir = Path("/Volumes/LUCY DISK/mia/Project 1/example test/test_images")
anchor_points_dir = Path("/Volumes/LUCY DISK/mia/Project 1/example test/anchor_images") # same folder as anchor images

grouping_results_csv = Path("/Users/lucywu/mia-final-1/grouping_results.csv") # GENERATED from assign_anchors.ipynb
submission_dir = Path("/Users/lucywu/mia-final-1/submission_step2") # GENERATED

# Matching / geometry params
ratio_thresh = 0.75
ransac_thresh = 3.0
nfeatures = 4000

assert anchors_dir.exists(), f"Missing anchors_dir: {anchors_dir}"
assert tests_dir.exists(), f"Missing tests_dir: {tests_dir}"
assert grouping_results_csv.exists(), f"Missing grouping_results_csv: {grouping_results_csv}"
assert anchor_points_dir.exists(), f"Missing anchor_points_dir: {anchor_points_dir}"

submission_dir.mkdir(parents=True, exist_ok=True)
(submission_dir / "correspondence").mkdir(parents=True, exist_ok=True)
(submission_dir / "warped_coordinates").mkdir(parents=True, exist_ok=True)

print("submission_dir:", submission_dir)

submission_dir: /Users/lucywu/mia-final-1/submission_step2


In [8]:
def to_id(name: str, prefix: str) -> str:
    stem = Path(name).stem
    m = re.search(r"(\d+)", stem)
    if m:
        return f"{prefix}_{int(m.group(1)):02d}"
    return stem


def preprocess_for_matching(img_bgr: np.ndarray) -> np.ndarray:
    if img_bgr is None:
        raise ValueError("Failed to read image")
    green = img_bgr[:, :, 1]
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    return clahe.apply(green)


def estimate_homography(anchor_img_bgr: np.ndarray, test_img_bgr: np.ndarray, ratio: float, ransac: float, nfeatures: int) -> Tuple[np.ndarray, Dict[str, float]]:
    anchor = preprocess_for_matching(anchor_img_bgr)
    test = preprocess_for_matching(test_img_bgr)

    sift = cv2.SIFT_create(nfeatures=nfeatures)
    kp_a, des_a = sift.detectAndCompute(anchor, None)
    kp_t, des_t = sift.detectAndCompute(test, None)

    if des_a is None or des_t is None or len(des_a) < 4 or len(des_t) < 4:
        raise RuntimeError("Not enough descriptors for homography")

    matcher = cv2.FlannBasedMatcher(dict(algorithm=1, trees=5), dict(checks=80))
    knn = matcher.knnMatch(des_a, des_t, k=2)

    good = []
    for pair in knn:
        if len(pair) < 2:
            continue
        m, n = pair
        if m.distance < ratio * n.distance:
            good.append(m)

    if len(good) < 4:
        raise RuntimeError(f"Insufficient good matches: {len(good)}")

    pts_a = np.float32([kp_a[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    pts_t = np.float32([kp_t[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)

    H, inlier_mask = cv2.findHomography(pts_a, pts_t, cv2.RANSAC, ransac)
    if H is None or inlier_mask is None:
        raise RuntimeError("Homography estimation failed")

    inliers = int(inlier_mask.ravel().sum())
    stats = {
        "num_keypoints_anchor": len(kp_a),
        "num_keypoints_test": len(kp_t),
        "num_good_matches": len(good),
        "num_inliers": inliers,
        "inlier_ratio": inliers / max(len(good), 1),
    }
    return H, stats


def transform_points(points_xy: np.ndarray, H: np.ndarray) -> np.ndarray:
    pts = points_xy.astype(np.float32).reshape(-1, 1, 2)
    warped = cv2.perspectiveTransform(pts, H).reshape(-1, 2)
    return warped


def load_anchor_points_csv(anchor_name: str, points_dir: Path) -> pd.DataFrame:
    stem = Path(anchor_name).stem
    candidates = [
        points_dir / f"{stem}.csv",
        points_dir / f"{stem}_coordinates.csv",
        points_dir / f"{stem}_sampled_pixels.csv",
        points_dir / f"{stem}_sampled_points.csv",
        points_dir / f"{stem}_points.csv",
    ]

    if stem.startswith("anchor_"):
        suffix = stem.replace("anchor_", "")
        candidates.extend([
            points_dir / f"anchor_{suffix}.csv",
            points_dir / f"anchor_{suffix}_coordinates.csv",
            points_dir / f"anchor_{suffix}_sampled_pixels.csv",
        ])

    for p in candidates:
        if p.name.startswith("._"):
            continue
        if p.exists():
            df = pd.read_csv(p)
            cols = [c.lower().strip() for c in df.columns]
            if len(cols) < 2:
                raise ValueError(f"Anchor points csv has <2 columns: {p}")
            x_col = df.columns[0]
            y_col = df.columns[1]
            out = pd.DataFrame({"x": df[x_col].astype(float), "y": df[y_col].astype(float)})
            if len(out) != 100:
                print(f"Warning: expected 100 points but found {len(out)} in {p.name}")
            return out

    csvs = [p for p in sorted(points_dir.glob("*.csv")) if not p.name.startswith("._")]
    names = [c.name for c in csvs[:10]]
    raise FileNotFoundError(
        f"No anchor-point CSV found for {anchor_name}. Checked {len(candidates)} filename patterns in {points_dir}."
        f" Example csv files here: {names}"
    )

In [ ]:
grouping_df = pd.read_csv(grouping_results_csv)
required = {"test_image", "anchor_image"}
missing_cols = required - set(grouping_df.columns)
if missing_cols:
    raise ValueError(f"grouping_results.csv missing columns: {missing_cols}")

grouping_submission_rows: List[Dict[str, str]] = []

for _, row in grouping_df.iterrows():
    test_name = str(row["test_image"]).strip()
    anchor_name = str(row["anchor_image"]).strip()

    test_path = tests_dir / test_name
    anchor_path = anchors_dir / anchor_name

    if not test_path.exists():
        raise FileNotFoundError(f"Missing test image: {test_path}")
    if not anchor_path.exists():
        raise FileNotFoundError(f"Missing anchor image: {anchor_path}")

    anchor_pts_df = load_anchor_points_csv(anchor_name, anchor_points_dir)

    anchor_img_bgr = cv2.imread(str(anchor_path), cv2.IMREAD_COLOR)
    test_img_bgr = cv2.imread(str(test_path), cv2.IMREAD_COLOR)

    H, _stats = estimate_homography(
        anchor_img_bgr=anchor_img_bgr,
        test_img_bgr=test_img_bgr,
        ratio=ratio_thresh,
        ransac=ransac_thresh,
        nfeatures=nfeatures,
    )

    points_xy = anchor_pts_df[["x", "y"]].to_numpy(dtype=np.float32)
    warped_xy = transform_points(points_xy, H)

    test_id = to_id(test_name, "test")
    anchor_id = to_id(anchor_name, "anchor")

    warped_out = pd.DataFrame({"x": warped_xy[:, 0], "y": warped_xy[:, 1]})
    warped_csv_path = submission_dir / "warped_coordinates" / f"{test_id}_warped_coordinates.csv"
    warped_out.to_csv(warped_csv_path, index=False, float_format="%.2f")

    grouping_submission_rows.append({"test_id": test_id, "anchor_id": anchor_id})

grouping_submission_df = pd.DataFrame(grouping_submission_rows).sort_values("test_id")
grouping_submission_df.to_csv(submission_dir / "correspondence" / "grouping.csv", index=False)

print(f"Saved grouping file: {submission_dir / 'correspondence' / 'grouping.csv'}")
print(f"Saved warped coordinate files in: {submission_dir / 'warped_coordinates'}")
print("\nPreview:")
display(grouping_submission_df.head())


Saved grouping file: /Users/lucywu/mia-final-1/submission_step2/correspondence/grouping.csv
Saved warped coordinate files in: /Users/lucywu/mia-final-1/submission_step2/warped_coordinates
Saved debug metrics: /Users/lucywu/mia-final-1/submission_step2/warp_debug_metrics.csv

Preview:


,test_id,anchor_id
0,test_01,anchor_04
1,test_02,anchor_02
2,test_03,anchor_01
3,test_04,anchor_01
4,test_05,anchor_05


,test_image,anchor_image,test_id,anchor_id,num_keypoints_anchor,num_keypoints_test,num_good_matches,num_inliers,inlier_ratio
0,test_01.tiff,anchor_04.tiff,test_01,anchor_04,2424,358,240,45,0.187500
1,test_02.tiff,anchor_02.tiff,test_02,anchor_02,1094,1156,359,33,0.091922
2,test_03.tiff,anchor_01.tiff,test_03,anchor_01,4000,4000,1812,132,0.072848
3,test_04.tiff,anchor_01.tiff,test_04,anchor_01,4000,3670,2274,368,0.161829
4,test_05.tiff,anchor_05.tiff,test_05,anchor_05,2358,1796,601,77,0.128120


## Optional evaluation (use for testing)

In [12]:
import subprocess
import sys

gt_dir = Path('/Volumes/LUCY DISK/mia/Project 1/example test/ground_truth')
subprocess.run([
    sys.executable,
    '/Users/lucywu/mia-final-1/eval.py',
    '--pred_dir', str(submission_dir),
    '--gt_dir', str(gt_dir),
    '--skip', 'segmentation',
    '-v',
], check=True)

  MIA 2026 Project 1 — Evaluation Results

  Task 1: Anchor Assignment (Grouping)
------------------------------------------------------------------------
  Accuracy:  0.8000  (20/25)

  Incorrect assignments (5):
    test_05:  predicted=anchor_05,  correct=anchor_04
    test_07:  predicted=anchor_04,  correct=anchor_05
    test_08:  predicted=anchor_04,  correct=anchor_05
    test_16:  predicted=anchor_04,  correct=anchor_05
    test_22:  predicted=anchor_05,  correct=anchor_04

  Task 2: Warped Coordinate Prediction (Registration)
------------------------------------------------------------------------
  MSE:       258687.8470
  Evaluated: 25/50 test images

  Per-image MSE:
    test_01:  MSE = 11607.4790
    test_02:  MSE = 25462.8223
    test_03:  MSE = 1383.2263
    test_04:  MSE = 919.3477
    test_05:  MSE = 1281958.8553
    test_06:  MSE = 2432.8909
    test_07:  MSE = 1445533.2257
    test_08:  MSE = 1253909.3558
    test_09:  MSE = 8411.7907
    test_10:  MSE = 81361.5166
   

CompletedProcess(args=['/Users/lucywu/quantum/miniconda3/envs/mia-final-1/bin/python', '/Users/lucywu/mia-final-1/eval.py', '--pred_dir', '/Users/lucywu/mia-final-1/submission_step2', '--gt_dir', '/Volumes/LUCY DISK/mia/Project 1/example test/ground_truth', '--skip', 'segmentation', '-v'], returncode=0)